# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LaibaSabir1/flyrank-ml-internship-laiba_sabir/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/LaibaSabir1/flyrank-ml-internship-laiba_sabir"
REPO_DIR = "flyrank-ml-internship-laiba_sabir"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{df.shape[0]:,} pages | declining rate: {df['is_declining_label'].mean():.3f}")


30,000 pages | declining rate: 0.542


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Two signals I'm checking before I trust them:**

1. **Staleness** (`days_since_last_update`) — this is the signal behind FlyRank's real
   *refresh* flags. My starting intuition (carried over from `notebooks/02`, the
   `stale x visible` hand rule) was: "a page that hasn't been touched in a while and is
   still getting traffic is worth reviewing." Before I bake that into a score, I check
   whether staleness actually goes with decline in this data.
2. **CTR vs. position** — this is the signal behind FlyRank's real *CTR-fix* logic
   (`low_ctr_visible_page` in the reference pipeline). A page ranking well (top 3 / page 1)
   but still earning a low click-through rate is a classic "the snippet/title isn't
   pulling its weight" signal.

I score each with a bucket table (n printed) and a one-word verdict: **CONFIRMED / OPPOSITE
/ MIXED / FALSE**. A verdict threshold of ±0.05 on the decline-rate gap, with a 50-row floor
per bucket, keeps me from calling noise a signal.

In [2]:
N_FLOOR = 50
DIFF_THRESHOLD = 0.05

def verdict(diff: float, n_a: int, n_b: int) -> str:
    if n_a < N_FLOOR or n_b < N_FLOOR:
        return "INSUFFICIENT DATA"
    if diff > DIFF_THRESHOLD:
        return "CONFIRMED"
    if diff < -DIFF_THRESHOLD:
        return "OPPOSITE"
    return "MIXED"

In [3]:
# --- Signal 1: staleness -> decline rate (behind FlyRank's refresh flags) ---
staleness_table = (
    df.groupby("freshness_tier")["is_declining_label"]
    .agg(decline_rate="mean", n="count")
    .round(3)
)
print("Signal 1 — staleness (freshness_tier) vs decline rate\n")
print(staleness_table)

stale_mask = df["days_since_last_update"] >= 180
stale_rate = df.loc[stale_mask, "is_declining_label"].mean()
fresh_rate = df.loc[~stale_mask, "is_declining_label"].mean()
n_stale, n_fresh = int(stale_mask.sum()), int((~stale_mask).sum())
diff_1 = stale_rate - fresh_rate
verdict_1 = verdict(diff_1, n_stale, n_fresh)

print(f"\nStale  (days_since_last_update >= 180, n={n_stale:,}):  decline rate {stale_rate:.3f}")
print(f"Fresh  (days_since_last_update <  180, n={n_fresh:,}):  decline rate {fresh_rate:.3f}")
print(f"Difference: {diff_1:+.3f}   ->   Verdict: {verdict_1}")

Signal 1 — staleness (freshness_tier) vs decline rate

                decline_rate      n
freshness_tier                     
0-30                   0.511  20480
181+                   0.471    174
31-90                  0.589    175
91-180                 0.611   9171

Stale  (days_since_last_update >= 180, n=174):  decline rate 0.471
Fresh  (days_since_last_update <  180, n=29,826):  decline rate 0.542
Difference: -0.071   ->   Verdict: OPPOSITE


**Verdict on staleness: OPPOSITE.** Stale pages in this slice decline *less* often than
fresh ones (`freshness_tier` shows the same shape: `0-30` and `91-180` both sit close to or
above the fresh end, and the raw `>=180d` cut comes in *below* the base rate). That
overturns my carried-over intuition — a clearly-explained negative here is exactly the kind
of thing this check is supposed to catch. **I will not gate my rule on staleness.** It stays
useful as a reason code annotation later, but not as a trigger.


In [4]:
# --- Signal 2: CTR vs. position -> decline rate (behind FlyRank's CTR-fix logic) ---
strong_position = df["position_tier"].isin(["top_3", "page_1"])
visible = df["impressions_90d"] >= 500
candidate_pool = df[strong_position & visible]

low_ctr = candidate_pool["ctr"] < 0.5
ctr_table = (
    candidate_pool.assign(ctr_bucket=low_ctr.map({True: "ctr < 0.5%", False: "ctr >= 0.5%"}))
    .groupby("ctr_bucket")["is_declining_label"]
    .agg(decline_rate="mean", n="count")
    .round(3)
)
print("Signal 2 — CTR vs. position (top_3 / page_1, impressions_90d >= 500) vs decline rate\n")
print(ctr_table)

low_ctr_rate = candidate_pool.loc[low_ctr, "is_declining_label"].mean()
high_ctr_rate = candidate_pool.loc[~low_ctr, "is_declining_label"].mean()
n_low, n_high = int(low_ctr.sum()), int((~low_ctr).sum())
diff_2 = low_ctr_rate - high_ctr_rate
verdict_2 = verdict(diff_2, n_low, n_high)

print(f"\nLow CTR  (< 0.5%, n={n_low:,}):  decline rate {low_ctr_rate:.3f}")
print(f"OK  CTR  (>= 0.5%, n={n_high:,}):  decline rate {high_ctr_rate:.3f}")
print(f"Difference: {diff_2:+.3f}   ->   Verdict: {verdict_2}")

Signal 2 — CTR vs. position (top_3 / page_1, impressions_90d >= 500) vs decline rate

             decline_rate     n
ctr_bucket                     
ctr < 0.5%          0.626  5932
ctr >= 0.5%         0.451  1590

Low CTR  (< 0.5%, n=5,932):  decline rate 0.626
OK  CTR  (>= 0.5%, n=1,590):  decline rate 0.451
Difference: +0.175   ->   Verdict: CONFIRMED


**Verdict on CTR-vs-position: CONFIRMED.** Among pages that are both well-ranked (top 3 or
page 1) and visible (>=500 impressions/90d), the ones under-earning on CTR (<0.5%) decline
noticeably more often, on a comfortable sample size in both buckets. This is the real signal
FlyRank's own CTR-fix flag leans on, and it holds up here.

**My rule (plain words):** *"A page that ranks well, gets real traffic, but still earns a
low click-through rate is worth reviewing for a CTR fix first."* One trigger, one score, one
reason code, one action label:

- **Trigger:** `position_tier` in `{top_3, page_1}` AND `impressions_90d >= 500` AND
  `ctr < 0.5`
- **Score:** `impressions_90d` when triggered, else `0` — bigger visible pages with the same
  problem outrank smaller ones
- **Reason code:** `ctr_gap_at_strong_position` (single code; `not_flagged` otherwise)
- **Action:** `refresh_and_review_ctr` when triggered, else `monitor`

I dropped staleness from the trigger entirely, per Signal 1 above — including it would have
added a condition that this data says works *against* my rule, not for it.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
STRONG_POSITION_TIERS = {"top_3", "page_1"}
VISIBLE_IMPRESSIONS_FLOOR = 500
CTR_THRESHOLD = 0.5  # percent — remember ctr is already a 0-100 scale, not a fraction

df["is_strong_position"] = df["position_tier"].isin(STRONG_POSITION_TIERS)
df["is_visible"] = df["impressions_90d"] >= VISIBLE_IMPRESSIONS_FLOOR
df["is_low_ctr"] = df["ctr"] < CTR_THRESHOLD

flagged = df["is_strong_position"] & df["is_visible"] & df["is_low_ctr"]

df["baseline_score"] = df["impressions_90d"].where(flagged, 0)
df["reason_code"] = flagged.map({True: "ctr_gap_at_strong_position", False: "not_flagged"})
df["suggested_action"] = flagged.map({True: "refresh_and_review_ctr", False: "monitor"})
df["baseline_rank"] = df["baseline_score"].rank(method="first", ascending=False).astype(int)

print(f"Flagged rows: {int(flagged.sum()):,} of {len(df):,}")
print(f"Flagged decline rate:     {df.loc[flagged, 'is_declining_label'].mean():.3f}")
print(f"Not-flagged decline rate: {df.loc[~flagged, 'is_declining_label'].mean():.3f}")
print(f"Base rate (everyone):     {df['is_declining_label'].mean():.3f}")

Flagged rows: 5,932 of 30,000
Flagged decline rate:     0.626
Not-flagged decline rate: 0.521
Base rate (everyone):     0.542


In [6]:
output_columns = [
    "content_id", "client_id", "baseline_rank", "baseline_score",
    "reason_code", "suggested_action", "is_declining_label",
    "impressions_90d", "clicks_90d", "avg_position", "position_tier", "ctr",
    "days_since_last_update", "content_age_days", "trend_direction",
]

queue = df[output_columns].sort_values("baseline_rank").reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(queue_path, index=False)
print(f"Wrote {len(queue):,} rows to {queue_path}")

Wrote 30,000 rows to work/outputs/baseline_action_score.csv


In [7]:
def precision_at_k(ranked_frame: pd.DataFrame, k: int) -> float:
    return ranked_frame.head(k)["is_declining_label"].mean()

base_rate = df["is_declining_label"].mean()
for k in (10, 20, 50):
    print(f"Precision@{k}: {precision_at_k(queue, k):.3f}   (base rate: {base_rate:.3f})")

Precision@10: 0.600   (base rate: 0.542)
Precision@20: 0.500   (base rate: 0.542)
Precision@50: 0.420   (base rate: 0.542)


Reading this honestly: Precision@10 clears the base rate comfortably, but by Precision@50
the rule is basically at (or slightly below) the base rate. That's not a bug — the trigger
picks a genuinely higher-decline-rate *pool* (0.626 vs. base 0.542), but within that pool I'm
only breaking ties by raw `impressions_90d`, and impressions alone don't concentrate the
declining pages at the very top. That's a real weakness of this baseline, and it's exactly
the kind of gap a learned model (Week 5) should be able to close by using more than one
signal to rank *within* the flagged pool.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
top20 = queue.head(20).reset_index(drop=True)
top20[["baseline_rank", "impressions_90d", "avg_position", "position_tier", "ctr",
       "days_since_last_update", "trend_direction", "is_declining_label"]]

,baseline_rank,impressions_90d,avg_position,position_tier,ctr,days_since_last_update,trend_direction,is_declining_label
0,1,517715,4.2,page_1,0.14,104,down,1
1,2,517109,5.4,page_1,0.25,22,stable,0
2,3,509252,2.5,top_3,0.15,20,down,1
3,4,463103,2.3,top_3,0.41,20,down,1
4,5,416180,4.0,page_1,0.23,22,down,1
5,6,345111,5.4,page_1,0.21,20,up,0
6,7,309910,5.6,page_1,0.16,104,down,1
7,8,295097,7.3,page_1,0.05,104,stable,0
8,9,272144,2.3,top_3,0.03,20,up,0
9,10,236803,4.4,page_1,0.26,20,down,1


In [9]:
def what_would_make_it_wrong(row: pd.Series) -> str:
    if row["trend_direction"] not in ("down",):
        return (
            f"trend_direction is '{row['trend_direction']}', not 'down' — the CTR gap is real "
            "but this page isn't actually declining, so a refresh may not be the right fix"
        )
    if row["ctr"] == 0:
        return "ctr is exactly 0 — worth a manual check that clicks are being tracked at all before assuming it's a content problem"
    return (
        "if a sibling page absorbed this page's clicks (consolidation) or a competitor "
        "changed the SERP layout, the CTR gap isn't this page's fault and a refresh won't fix it"
    )

for _, row in top20.iterrows():
    print(f"Rank {int(row['baseline_rank']):>2}  |  action: {row['suggested_action']}")
    print(
        f"   why: {row['reason_code']} — {row['position_tier']} position "
        f"(avg {row['avg_position']}), {row['impressions_90d']:,} impressions/90d, "
        f"ctr {row['ctr']:.2f}%, actual trend = {row['trend_direction']}"
    )
    print(f"   what would make it wrong: {what_would_make_it_wrong(row)}")
    print()

Rank  1  |  action: refresh_and_review_ctr
   why: ctr_gap_at_strong_position — page_1 position (avg 4.2), 517,715 impressions/90d, ctr 0.14%, actual trend = down
   what would make it wrong: if a sibling page absorbed this page's clicks (consolidation) or a competitor changed the SERP layout, the CTR gap isn't this page's fault and a refresh won't fix it

Rank  2  |  action: refresh_and_review_ctr
   why: ctr_gap_at_strong_position — page_1 position (avg 5.4), 517,109 impressions/90d, ctr 0.25%, actual trend = stable
   what would make it wrong: trend_direction is 'stable', not 'down' — the CTR gap is real but this page isn't actually declining, so a refresh may not be the right fix

Rank  3  |  action: refresh_and_review_ctr
   why: ctr_gap_at_strong_position — top_3 position (avg 2.5), 509,252 impressions/90d, ctr 0.15%, actual trend = down
   what would make it wrong: if a sibling page absorbed this page's clicks (consolidation) or a competitor changed the SERP layout, the CTR gap 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
weak_top20 = top20[top20["trend_direction"] != "down"]
print(f"{len(weak_top20)} of the top 10 are NOT actually declining (trend_direction != 'down'):\n")
weak_top20[["baseline_rank", "trend_direction", "impressions_90d", "avg_position", "ctr"]]

10 of the top 10 are NOT actually declining (trend_direction != 'down'):



,baseline_rank,trend_direction,impressions_90d,avg_position,ctr
1,2,stable,517109,5.4,0.25
5,6,up,345111,5.4,0.21
7,8,stable,295097,7.3,0.05
8,9,up,272144,2.3,0.03
10,11,stable,223271,7.8,0.03
11,12,stable,213963,4.7,0.10
12,13,stable,211366,5.1,0.41
15,16,stable,201584,5.8,0.24
16,17,stable,201111,5.7,0.11
17,18,stable,198671,5.6,0.18


In [11]:
# Leakage check — the rule must only use pre-decision, observable signals.
inputs_used = {"position_tier", "impressions_90d", "ctr"}
label_derived = {"trend_direction", "trend_pct"}
product_flags_not_in_dataset = {"health_score", "needs_ctr_fix", "is_quick_win", "priority_score", "action_type"}

assert inputs_used.isdisjoint(label_derived), "Leakage: a label-derived column is driving the score"
assert inputs_used.isdisjoint(product_flags_not_in_dataset), "Leakage: a product decision flag is driving the score"

print("Leakage check passed.")
print(f"Rule inputs:        {sorted(inputs_used)}")
print(f"Label-derived (never used): {sorted(label_derived)}")
print("These columns aren't even in the starter dataset, so they can't leak in: "
      f"{sorted(product_flags_not_in_dataset)}")

Leakage check passed.
Rule inputs:        ['ctr', 'impressions_90d', 'position_tier']
Label-derived (never used): ['trend_direction', 'trend_pct']
These columns aren't even in the starter dataset, so they can't leak in: ['action_type', 'health_score', 'is_quick_win', 'needs_ctr_fix', 'priority_score']


**Reading the weak picks:** a handful of the top 10 are ranked `stable` or `up`, not
`down` — the CTR-gap-at-strong-position signal is real *on average* (0.626 vs. 0.542
decline rate across the whole flagged pool), but it's a probabilistic signal, not a
guarantee for any single page. That's consistent with the honest framing this internship
asks for: this baseline is a **review-priority aid**, not a decline detector for any one
row. A reviewer should still glance at `trend_direction` and recent CTR history before
acting on any single flagged page.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.